# Building a Reproducible Mental Health Data Ecosystem: The Kilifi County, Kenya FAIR² Model Exploration with `mlcroissant`

This notebook demonstrates how to load and explore the FAIR² Mental Health dataset using the [`mlcroissant`](https://github.com/mlcommons/croissant) library, following best practices for referencing entities by their Croissant `@id` metadata.

### Dataset Source
The dataset source is defined by a Croissant schema at:

`https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json`

In [ ]:
# Install mlcroissant if not already available
!pip install mlcroissant

## 1. Data Loading

Load metadata and records from the dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import json
import warnings
warnings.filterwarnings('ignore')

# Define the Croissant metadata schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json'

# Load the dataset
dataset = mlc.Dataset(croissant_url)

# View high-level metadata (title and description)
meta = dataset.metadata
print('Dataset Title:', meta.name)
print('Description:', meta.description)

## 2. Data Overview

Identify available record sets, fields, and reference their `@id` values as specified by the Croissant schema.

In [ ]:
# List all record sets and show their @id and associated fields by @id
record_sets = list(dataset.metadata.record_sets)

print('Available Record Sets:')
all_record_set_ids = []
for recset in record_sets:
    print(f"- RecordSet Name: {recset.name}")
    print(f"  @id: {recset.id}")
    all_record_set_ids.append(recset.id)
    print("  Fields:")
    for field in recset.fields:
        print(f"    - {field.name} (@id: {field.id}, type: {field.data_type})")
    print()

# As an example, print a few records from the first record set
if record_sets:
    sample_record_set = record_sets[0]
    print(f"Sample records from RecordSet '{sample_record_set.name}' (@id: {sample_record_set.id}):")
    for ix, record in enumerate(dataset.records(record_set=sample_record_set.id)):
        print(json.dumps(record, indent=2))
        if ix >= 1:
            break


## 3. Data Extraction

Load data from all record sets into pandas DataFrames for exploration. All references use record set and field `@id` (as per the Croissant schema).

In [ ]:
# Load all record sets into DataFrames by their @id
dfs = {}

print('Loading all record sets into DataFrames:')
for recset in record_sets:
    recs = list(dataset.records(record_set=recset.id))
    if recs:
        df = pd.DataFrame(recs)
        dfs[recset.id] = df
        print(f"- {recset.name} @id: {recset.id}, shape: {df.shape}")
    else:
        print(f"- {recset.name} @id: {recset.id}: No records found!")

# List columns of the primary record set (assume the first one for demo)
if dfs:
    main_record_set_id = list(dfs.keys())[0]
    print(f"\nColumns in main record set (@id: {main_record_set_id}):")
    print(dfs[main_record_set_id].columns.tolist())
    display(dfs[main_record_set_id].head())

## 4. Exploratory Data Analysis (EDA)

Perform basic data checks and transformations, such as numeric filtering and normalization. Reference all columns using their full Croissant `@id` as shown in the overview above.

We choose GAD-7 total score as our numeric field for demonstration. The group field is 'demographic:gender', if available as a field.

In [ ]:
# Find the @id for the GAD-7 total field and a demographic field like gender.

main_df = dfs[main_record_set_id]

# Show available columns (~@ids) for EDA selection
print("Fields in main record set (by column/@id):")
print(main_df.columns.tolist())

# Attempt to select common field IDs for GAD-7 and gender
# This block assumes common naming (update to match actual field @ids above if different)
gad7_field_candidates = [c for c in main_df.columns if 'gad' in c.lower() and ('total' in c.lower() or 'score' in c.lower())]
gender_field_candidates = [c for c in main_df.columns if 'gender' in c.lower()]

if gad7_field_candidates:
    gad7_field_id = gad7_field_candidates[0]  # Croissant @id for GAD-7 total
    print(f"Using GAD-7 Total field: {gad7_field_id}")
else:
    gad7_field_id = main_df.select_dtypes(include='number').columns[0]
    print(f"No GAD-7 field found, using first numeric field: {gad7_field_id}")

if gender_field_candidates:
    group_field_id = gender_field_candidates[0]  # Croissant @id for gender
    print(f"Grouping by gender: {group_field_id}")
else:
    group_field_id = None

threshold = 10  # Example filter value
filtered_df = main_df[main_df[gad7_field_id] > threshold].copy()
print(f"\nFiltered records where {gad7_field_id} > {threshold} (n={len(filtered_df)}):")
display(filtered_df.head())

# Normalize GAD-7 scores
filtered_df[f"{gad7_field_id}_normalized"] = (filtered_df[gad7_field_id] - filtered_df[gad7_field_id].mean()) / filtered_df[gad7_field_id].std()
print(f"\nNormalized {gad7_field_id} for filtered records:")
display(filtered_df[[gad7_field_id, f"{gad7_field_id}_normalized"]].head())

# Grouping by gender if available
if group_field_id and group_field_id in filtered_df.columns:
    grouped = filtered_df.groupby(group_field_id)[gad7_field_id].mean().to_frame('Mean GAD-7 score')
    print('\nMean GAD-7 score by Gender:')
    display(grouped)


## 5. Visualization

Visualize the distribution of GAD-7 scores and how they vary by demographic fields such as gender (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if gad7_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(main_df[gad7_field_id].dropna(), bins=15, kde=True)
    plt.title(f'Distribution of GAD-7 scores ({gad7_field_id})')
    plt.xlabel('GAD-7 Score')
    plt.ylabel('Frequency')
    plt.show()

if group_field_id and gad7_field_id in main_df.columns:
    plt.figure(figsize=(8,4))
    sns.boxplot(x=main_df[group_field_id], y=main_df[gad7_field_id])
    plt.title('GAD-7 Scores by Gender')
    plt.xlabel('Gender')
    plt.ylabel('GAD-7 Score')
    plt.show()


## 6. Conclusion

- The dataset was successfully loaded using the Croissant schema and the `mlcroissant` library.
- Data exploration included extraction of all main tables, identification and normalization of key numeric fields such as GAD-7 total scores, and subgroup analysis by demographic attributes (when available).
- Visualizations show the distribution of mental health survey scores and demographic splits, enabling further research and statistical analysis.

For more advanced usage or documentation, see the [`mlcroissant` documentation](https://github.com/mlcommons/croissant) and the [FAIR² dataset description](https://sen.science/doi/10.71728/senscience.vcs2-05nj/fair2.json).